# Overview for our dataset

## Import parquet dataset (both train_raw and processed)

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

parquet_path = Path("./data/qasper/processed/processed.parquet")
parquet_raw_path = Path("./data/qasper/raw/train.parquet")
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")
if not parquet_raw_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_raw_path}")

print(f"Using parquet: {parquet_path}")

Using parquet: data/qasper/processed/processed.parquet


In [2]:
table = pq.read_table(parquet_path)
df = table.to_pandas()

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

schema_df = df.dtypes.astype(str).rename("dtype").to_frame()
schema_df.head(20)

Rows: 4,149
Columns: 5


,dtype
question,str
context,str
abstract,str
answer,str
source,str


In [3]:
table = pq.read_table(parquet_raw_path)
df_raw = table.to_pandas()

print(f"Rows: {len(df_raw):,}")
print(f"Columns: {df_raw.shape[1]}")

schema_df_raw = df.dtypes.astype(str).rename("dtype").to_frame()
schema_df_raw.head(20)

Rows: 2,593
Columns: 17


,dtype
question,str
context,str
abstract,str
answer,str
source,str


In [4]:
display(df.head(5))

missing = df.isna().sum().sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing": missing,
    "missing_pct": (missing / len(df) * 100).round(2),
})

print("\nTop missing-value columns:")
missing_summary.head(10)

,question,context,abstract,answer,source
0,How big is the ANTISCAM dataset?,To enrich available non-collaborative task dat...,End-to-end task-oriented dialog models have ac...,"3,044 sentences in 100 dialogs",1911.10742
1,How is intent annotated?,To enrich publicly available non-collaborative...,End-to-end task-oriented dialog models have ac...,using a role-playing task on the Amazon Mechan...,1911.10742
2,What are the baselines outperformed by this work?,We compare MISSA mainly with two baseline mode...,End-to-end task-oriented dialog models have ac...,TransferTransfo and Hybrid,1911.10742
3,What are the evaluation metrics and criteria u...,Experiments ::: Automatic Evaluation Metrics\n...,End-to-end task-oriented dialog models have ac...,Automatic evaluation metrics (Perplexity (PPl)...,1911.10742
4,What previous methods do they compare against?,To evaluate our new features for rumour detect...,Rumour detection is hard because the most accu...,They compare against two other methods that ap...,1611.06322



Top missing-value columns:


,missing,missing_pct
question,0,0.0
context,0,0.0
abstract,0,0.0
answer,0,0.0
source,0,0.0


In [8]:
finetune_df = pd.DataFrame({
    "input": df[["question", "context", "abstract"]].to_dict(orient="records"),
    "answer": df["answer"],
})

display(finetune_df.head())

,input,answer
0,{'question': 'How big is the ANTISCAM dataset?...,"3,044 sentences in 100 dialogs"
1,"{'question': 'How is intent annotated?', 'cont...",using a role-playing task on the Amazon Mechan...
2,{'question': 'What are the baselines outperfor...,TransferTransfo and Hybrid
3,{'question': 'What are the evaluation metrics ...,Automatic evaluation metrics (Perplexity (PPl)...
4,{'question': 'What previous methods do they co...,They compare against two other methods that ap...


In [ ]:
finetune_parquet_path = Path("./data/qasper/processed/finetune.parquet")
finetune_df.to_parquet(finetune_parquet_path, index=False)

print(f"Saved to: {finetune_parquet_path}")

In [5]:
display(df_raw.head(5))

missing_raw = df_raw.isna().sum().sort_values(ascending=False)
missing_raw_summary = pd.DataFrame({
    "missing_raw": missing,
    "missing_raw_pct": (missing / len(df_raw) * 100).round(2),
})

print("\nTop missing_raw-value columns:")
missing_raw_summary.head(10)

,qasper_id,question_id,question_text,answer_type,raw_answer,abstract,evidence,has_float_evidence,is_unanswerable,is_cleanable,annotator_count,annotators_agree_answersability,split,nlp_background,topic_background,paper_read,search_query
0,1909.00694,753990d0b621d390ed58f20c4d9e4f065f0dc672,What is the seed lexicon?,free_form,a vocabulary of positive and negative predicat...,Recognizing affective events that trigger posi...,[The seed lexicon consists of positive and neg...,False,False,True,2,True,train,two,unfamiliar,True,
1,1909.00694,9d578ddccc27dd849244d632dd0f6bf27348ad81,What are the results?,free_form,Using all data to train: AL -- BiGRU achieved ...,Recognizing affective events that trigger posi...,"[As for ${\rm Encoder}$, we compared two types...",True,False,True,1,True,train,two,unfamiliar,True,
2,1909.00694,02e4bf719b1a504e385c35c6186742e720bcb281,How are relations used to propagate polarity?,free_form,"based on the relation between events, the sugg...",Recognizing affective events that trigger posi...,"[In this paper, we propose a simple and effect...",False,False,True,2,True,train,two,unfamiliar,True,
3,1909.00694,44c4bd6decc86f1091b5fc0728873d9324cdde4e,How big is the Japanese data?,free_form,7000000 pairs of events were extracted from th...,Recognizing affective events that trigger posi...,"[As a raw corpus, we used a Japanese web corpu...",True,False,True,2,True,train,two,unfamiliar,True,
4,1909.00694,86abeff85f3db79cf87a8c993e5e5aa61226dc98,What are labels available in dataset for super...,extractive,negative positive,Recognizing affective events that trigger posi...,[Affective events BIBREF0 are events that typi...,False,False,True,1,True,train,zero,unfamiliar,True,



Top missing_raw-value columns:


,missing_raw,missing_raw_pct
question,0,0.0
context,0,0.0
abstract,0,0.0
answer,0,0.0
source,0,0.0


In [6]:
from datasets import Dataset
import json

# !! We start this from raw training data split
sft_df = df_raw.copy()

def pick_column(df, candidates, required=True):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f"None of the candidate columns were found: {candidates}")
    return None

context_col = pick_column(sft_df, ["evidence"], required=False)
question_col = pick_column(sft_df, ["question_text"])
answer_col = pick_column(sft_df, ["raw_answer"], required=False)
evidence_col = pick_column(sft_df, ["evidence"], required=False)

def format_evidence(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "none"
    if isinstance(value, list):
        cleaned = [str(v).strip() for v in value if str(v).strip()]
        return cleaned if cleaned else ["none"]
    text = str(value).strip()
    return [text] if text else ["none"]

def is_unanswerable(answer):
    if answer is None or (isinstance(answer, float) and pd.isna(answer)):
        return True
    normalized = str(answer).strip().lower()
    return normalized in {"", "unanswerable", "cannot answer", "not answerable"}

def build_response(row):
    answer_value = row.get(answer_col) if answer_col else None
    evidence_value = row.get(evidence_col) if evidence_col else None

    unanswerable = is_unanswerable(answer_value)
    payload = {
        "answerability": "unanswerable" if unanswerable else "answerable",
        "evidence": format_evidence(evidence_value),
        "answer": "unanswerable" if unanswerable else str(answer_value).strip(),
    }
    return json.dumps(payload, ensure_ascii=False)

# 1) Build SFT prompt / response fields
sft_df["response"] = sft_df.apply(build_response, axis=1)

sft_df["prompt"] = sft_df.apply(
    lambda r: (
        "You are a helpful scientific QA assistant. "
        "Answer the question based only on the provided paper content.\n\n"
        f"### Paper Context\n{(r.get(context_col) if context_col else 'not provided')}\n\n"
        f"### Question\n{r.get(question_col, '')}\n\n"
        "Return only valid JSON on a single line with exactly these keys:\n"
        '{"answerability":"answerable|unanswerable","evidence":["..."],"answer":"..."}'
    ),
    axis=1,
 )

# 2) Convert to Hugging Face Dataset
cols = ["prompt", "response"] + (["source"] if "source" in sft_df.columns else [])
hf_ds = Dataset.from_pandas(sft_df[cols], preserve_index=False)

print({
    "context_col": context_col,
    "question_col": question_col,
    "answer_col": answer_col,
    "evidence_col": evidence_col,
})
print(hf_ds)
display(sft_df[[question_col, "response"]].head(2))

/home/AGNDM/miniconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'context_col': 'evidence', 'question_col': 'question_text', 'answer_col': 'raw_answer', 'evidence_col': 'evidence'}
Dataset({
    features: ['prompt', 'response'],
    num_rows: 2593
})


,question_text,response
0,What is the seed lexicon?,"{""answerability"": ""answerable"", ""evidence"": [""..."
1,What are the results?,"{""answerability"": ""answerable"", ""evidence"": [""..."


## Notes
- If training with `Trainer` + causal LM, the `text` column is enough.
- If using chat templates, keep `prompt`/`response` and format during tokenization. (We are going to use this)

## Demonstration of User Prompt and Response

In [7]:
import json
import textwrap

sample_idx = 3
prompt_text = sft_df.loc[sample_idx, "prompt"]
response_text = sft_df.loc[sample_idx, "response"]

try:
    response_obj = json.loads(response_text)
    pretty_response = json.dumps(response_obj, indent=2, ensure_ascii=False)
except Exception:
    pretty_response = response_text

print("PROMPT")
print("=" * 80)
print(textwrap.fill(prompt_text, width=100, replace_whitespace=False))

print("\nRESPONSE (JSON)")
print("=" * 80)
print(pretty_response)

# quick tabular preview
display(
    sft_df[["prompt", "response"]]
    .head(0)
    .assign(response=lambda d: d["response"].apply(lambda x: json.dumps(json.loads(x), indent=2) if isinstance(x, str) and x.strip().startswith("{") else x))
)

PROMPT
You are a helpful scientific QA assistant. Answer the question based only on the provided paper
content.

### Paper Context
['As a raw corpus, we used a Japanese web corpus that was compiled
through the procedures proposed by BIBREF13. To extract event pairs tagged with discourse relations,
we used the Japanese dependency parser KNP and in-house postprocessing scripts BIBREF14. KNP used
hand-written rules to segment each sentence into what we conventionally called clauses (mostly
consecutive text chunks), each of which contained one main predicate. KNP also identified the
discourse relations of event pairs if explicit discourse connectives BIBREF4 such as “ので” (because)
and “のに” (in spite of) were present. We treated Cause/Reason (原因・理由) and Condition (条件) in the
original tagset BIBREF15 as Cause and Concession (逆接) as Concession, respectively. Here is an
example of event pair extraction.'
 'We constructed our seed lexicon consisting of 15 positive words
and 15 negative words, a

,prompt,response
